In [ ]:
from gui_core.io import load_config, save_config
from gui_core import folder_logic
import os

#LOAD Default Config
config = load_config()
config.main_folder = '' #Update main_folder
config.data_extension = '.tif' #Update for your file type

#Add groups / experimental conditions based on your main folder / subfolder organization
config.groups = folder_logic.find_valid_folders(config.main_folder, config.data_extension)
config.exp_condition = folder_logic.build_exp_condition(config.groups)

config.frame_rate = 20 #Adjust frame rate (integer value)
config.ops_path = r'' #Path to suite2p settings file (usually ops.npy)
config.experiment_duration = 180 #(Integer value) for seconds of experiment

#_____ANALYSIS PARAMETERS_____#
config.analysis_params.overwrite_suite2p = 'False'
config.analysis_params.skew_threshold = 1 #Fluorescence skew threshold for describing trace shape; set to 0 to turn off
config.analysis_params.compactness_threshold = 1.4 #Default value, set to 0 to turn off
config.analysis_params.peak_count_threshold = 1 #Number of peaks per synapse before synapse in included in analysis
config.analysis_params.peak_detection_threshold = 4.5 #Number of SD above MAD to detect peaks; for high SnR, values above 4.5 are possible and could be beneficial
config.analysis_params.Img_Overlay='max_proj'
config.analysis_params.use_suite2p_ROI_classifier=True
config.analysis_params.update_suite2p_iscell=False
config.analysis_params.return_decay_times=False

save_config(path = os.path.join(config.main_folder, 'config', 'config.json'), config=config)

In [ ]:
from analyze_suite2p.config_loader import load_json_config_file, load_json_dict
from analyze_suite2p import run_suite2p, suite2p_utility, analysis_utility
import json
main_folder = config.general_settings.main_folder

#Load JSON config file as SimpleNameSpace Object
config = load_json_config_file()
config_dict = load_json_dict()
#Run suite2p main
# run_suite2p.main(config_file=config)

#run_suite2p.main() parts
image_folders = run_suite2p.get_all_image_folders_in_path(config.general_settings.main_folder)
suite2p_samples = suite2p_utility.get_all_suite2p_outputs_in_path(folder_path=config.general_settings.main_folder, 
                                                                  file_ending='samples', 
                                                                  supress_printing=False)

import numpy as np
ops = np.load(config.general_settings.ops_path, allow_pickle=True).item()

run_suite2p.process_files_with_suite2p(image_folders, ops)


with open(os.path.join(main_folder, 'analysis_config.json'), 'w') as f:
        json.dump(config_dict, f, indent = 4)
        print(f"Analysis parameters saved in {main_folder} as analysis_config.json")
analysis_utility.generate_synapse_counts_and_summary_stats(main_folder)


In [ ]:
from analyze_suite2p import analysis_utility
from analyze_suite2p import config_loader

config = config_loader.load_json_config_file()

analysis_utility.translate_suite2p_outputs_to_csv(main_folder = config.general_settings.main_folder, 
                                                  config = config, 
                                                  check_for_iscell=False, update_iscell=True)
analysis_utility.process_spike_csvs_to_pkl(input_path=config.general_settings.main_folder)

analysis_utility.create_experiment_summary(main_folder=config.general_settings.main_folder)